In [13]:
from pathlib import Path
from pprint import pprint
from plant_pheno.inference import inat_client
from plant_pheno.data import DuckDbSQL, DuckDBAdapter

In [2]:

obs_id = [182806784]
rate = 10

TARGET_TABLE_NAME = "raw.inat_api"

OBSERVATIONS_FIELDS = {
    "photos" : True,
    "observation_photos" : True
}

In [ ]:
with DuckDBAdapter("/home/etienne/projects/inat-phenology-cv/data/cv_raw.duckdb") as con:
    sql_api = DuckDbSQL(con, Path("/home/etienne/projects/inat-phenology-cv/queries/api/"))
    sql_api.execute("create_api_raw_table", table_name=TARGET_TABLE_NAME)

    config = inat_client.EndpointConfig(
        "observations",
        id_param="id",
        write_empty_rows=True,
        fields = OBSERVATIONS_FIELDS,
        chunk_size= 200,
        per_page= 200,
        id_fields=['id']
    )

    pprint(config)
    
    fetcher = inat_client.RateLimiterFetcher(rate=rate, ignore_not_found=True)
    with inat_client.DuckDbWriter(con, TARGET_TABLE_NAME) as writer:
        client = inat_client.make_client(config, fetcher, writer)
        await client.execute(obs_id)
    

post
EndpointConfig(endpoint='observations',
               fields='(photos:!t,observation_photos:!t)',
               params={},
               per_page=200,
               chunk_size=200,
               id_param='id',
               api_version=2,
               url='https://api.inaturalist.org/v2/observations',
               id_fields=['id'],
               write_empty_rows=True)
